In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [4]:
spark = SparkSession.builder.appName("ETL_Pipeline").getOrCreate()

# Extract Data

In [26]:
orders_df = spark.read.csv("orders.csv", header=True, inferSchema=True)
products_df = spark.read.csv("products.csv", header=True, inferSchema=True)

In [27]:
orders_df.printSchema()
products_df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- order_date: string (nullable = true)
 |-- region: string (nullable = true)

root
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: integer (nullable = true)



In [28]:
orders_df.show(5)

+--------+----------+--------+----------+-------+
|order_id|product_id|quantity|order_date| region|
+--------+----------+--------+----------+-------+
|       1|        P8|       4|21-05-2023|Central|
|       2|        P7|    NULL|12-12-2023|  South|
|       2|       P15|       5|02-05-2023|  North|
|       3|        P5|       4|03-07-2023|   West|
|       3|       P13|       2|07-10-2023|  South|
+--------+----------+--------+----------+-------+
only showing top 5 rows


In [29]:
products_df.show()

+----------+-----------+-----+
|product_id|   category|price|
+----------+-----------+-----+
|        P1|Electronics|  200|
|        P2|Electronics|  150|
|        P3|  Furniture|  300|
|        P4|   Clothing|  120|
|        P5|    Kitchen|   80|
|        P6|     Sports|  250|
|        P7|Electronics|  180|
|        P8|  Furniture|  350|
|        P9|   Clothing|   90|
|       P10|    Kitchen|   60|
|       P11|     Sports|  200|
|       P12|Electronics|  220|
|       P13|  Furniture|  400|
|       P14|   Clothing|  110|
|       P15|       NULL|   70|
+----------+-----------+-----+



# Data Cleaning

In [30]:
orders_df.filter(F.col("quantity").isNull()).show()

+--------+----------+--------+----------+-------+
|order_id|product_id|quantity|order_date| region|
+--------+----------+--------+----------+-------+
|       2|        P7|    NULL|12-12-2023|  South|
|      24|       P14|    NULL|10-03-2023|Central|
|      30|        P7|    NULL|24-06-2023|   West|
|      33|        P8|    NULL|16-05-2023|Central|
|      37|       P11|    NULL|03-01-2023|   East|
|      41|        P5|    NULL|13-05-2023|Central|
|      47|       P11|    NULL|11-07-2023|  North|
|      68|        P7|    NULL|18-06-2023|   West|
|      91|        P3|    NULL|01-11-2023|   East|
|      93|        P1|    NULL|13-04-2023|Central|
|     102|       P11|    NULL|05-07-2023|Central|
|     103|       P11|    NULL|08-12-2023|  South|
|     105|        P8|    NULL|14-04-2023|   West|
|     128|        P6|    NULL|24-11-2023|Central|
|     135|       P13|    NULL|21-10-2023|Central|
|     136|        P5|    NULL|11-01-2023|  North|
|     137|        P7|    NULL|03-08-2023|  South|


In [31]:
orders_df = orders_df.fillna({"quantity": 1})

In [32]:
orders_df.show(5)

+--------+----------+--------+----------+-------+
|order_id|product_id|quantity|order_date| region|
+--------+----------+--------+----------+-------+
|       1|        P8|       4|21-05-2023|Central|
|       2|        P7|       1|12-12-2023|  South|
|       2|       P15|       5|02-05-2023|  North|
|       3|        P5|       4|03-07-2023|   West|
|       3|       P13|       2|07-10-2023|  South|
+--------+----------+--------+----------+-------+
only showing top 5 rows


In [33]:
products_df.filter(F.col("category").isNull()).show()

+----------+--------+-----+
|product_id|category|price|
+----------+--------+-----+
|       P15|    NULL|   70|
+----------+--------+-----+



In [34]:
products_df = products_df.fillna({"category": "Unknown"})

In [35]:
products_df.show()

+----------+-----------+-----+
|product_id|   category|price|
+----------+-----------+-----+
|        P1|Electronics|  200|
|        P2|Electronics|  150|
|        P3|  Furniture|  300|
|        P4|   Clothing|  120|
|        P5|    Kitchen|   80|
|        P6|     Sports|  250|
|        P7|Electronics|  180|
|        P8|  Furniture|  350|
|        P9|   Clothing|   90|
|       P10|    Kitchen|   60|
|       P11|     Sports|  200|
|       P12|Electronics|  220|
|       P13|  Furniture|  400|
|       P14|   Clothing|  110|
|       P15|    Unknown|   70|
+----------+-----------+-----+



In [36]:
orders_df.filter(F.col("quantity") <= 0).show()

+--------+----------+--------+----------+------+
|order_id|product_id|quantity|order_date|region|
+--------+----------+--------+----------+------+
+--------+----------+--------+----------+------+



In [37]:
products_df.filter(F.col("price") <= 0).show()

+----------+--------+-----+
|product_id|category|price|
+----------+--------+-----+
+----------+--------+-----+



In [38]:
products_df.groupBy("product_id").count().filter(F.col("count") > 1).show()

+----------+-----+
|product_id|count|
+----------+-----+
+----------+-----+



# Merging both tables using inner join

In [39]:
orders_df.join(
    products_df,
    on="product_id",
    how="left_anti"
).show()

+----------+--------+--------+----------+------+
|product_id|order_id|quantity|order_date|region|
+----------+--------+--------+----------+------+
+----------+--------+--------+----------+------+



In [40]:
merge_df = orders_df.join(products_df, on="product_id", how="inner")

In [41]:
merge_df = merge_df.withColumn("order_date", F.to_date(F.col("order_date"), "dd-MM-yyyy"))

In [42]:
merge_df.show(5)

+----------+--------+--------+----------+-------+-----------+-----+
|product_id|order_id|quantity|order_date| region|   category|price|
+----------+--------+--------+----------+-------+-----------+-----+
|        P8|       1|       4|2023-05-21|Central|  Furniture|  350|
|        P7|       2|       1|2023-12-12|  South|Electronics|  180|
|       P15|       2|       5|2023-05-02|  North|    Unknown|   70|
|        P5|       3|       4|2023-07-03|   West|    Kitchen|   80|
|       P13|       3|       2|2023-10-07|  South|  Furniture|  400|
+----------+--------+--------+----------+-------+-----------+-----+
only showing top 5 rows


# Add revenue column

In [43]:
merge_df = merge_df.withColumn("revenue", F.col("quantity") * F.col("price"))

In [44]:
merge_df.show(5)

+----------+--------+--------+----------+-------+-----------+-----+-------+
|product_id|order_id|quantity|order_date| region|   category|price|revenue|
+----------+--------+--------+----------+-------+-----------+-----+-------+
|        P8|       1|       4|2023-05-21|Central|  Furniture|  350|   1400|
|        P7|       2|       1|2023-12-12|  South|Electronics|  180|    180|
|       P15|       2|       5|2023-05-02|  North|    Unknown|   70|    350|
|        P5|       3|       4|2023-07-03|   West|    Kitchen|   80|    320|
|       P13|       3|       2|2023-10-07|  South|  Furniture|  400|    800|
+----------+--------+--------+----------+-------+-----------+-----+-------+
only showing top 5 rows


# Aggregation

In [45]:
total_revenue = merge_df.groupBy("category").agg(F.sum("revenue").alias("total_revenue")).orderBy(F.col("total_revenue").desc())

In [46]:
total_revenue.show()

+-----------+-------------+
|   category|total_revenue|
+-----------+-------------+
|  Furniture|       153050|
|Electronics|       114070|
|     Sports|        62100|
|   Clothing|        43940|
|    Kitchen|        17980|
|    Unknown|         9240|
+-----------+-------------+



# Add data to database

In [21]:
pip install sqlalchemy psycopg2-binary pandas

   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   --------------- ------------------------ 1.0/2.8 MB 5.0 MB/s eta 0:00:01
   -------------------------- ------------- 1.8/2.8 MB 4.6 MB/s eta 0:00:01
   -------------------------------------- - 2.6/2.8 MB 4.3 MB/s eta 0:00:01
   -------------------------------------- - 2.6/2.8 MB 4.3 MB/s eta 0:00:01
   ---------------------------------------- 2.8/2.8 MB 3.0 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: C:\Users\Udvab\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip


In [47]:
from sqlalchemy import create_engine

In [48]:
pandas_df = merge_df.toPandas()

In [54]:
import os
from dotenv import load_dotenv

In [51]:
username = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
host = os.getenv("DB_HOST")
port = os.getenv("DB_PORT")
database = os.getenv("DB_NAME")

engine = create_engine(
    f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}"
)

pandas_df.to_sql(
    name="sales_data",
    con=engine,
    if_exists="replace",
    index=False
)

print("Data successfully inserted into PostgreSQL!")

Data successfully inserted into PostgreSQL!
